<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/25_phase3_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/25_phase3_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 25 — Faz 3: uçtan uca fine-tuning (T4 GPU)

**Plan:** `docs/PHASE3_DESIGN.md` (D-042). İki bölüm var:
1. **SMK-002 (smoke test):** her kol için gerçek ağırlıklarla küçük bir eğitim. Kontrol edilenler: şekiller, kayıp, checkpoint'ten aynı sonuç, GPU belleği ve süre. Sonuç üretmez; rapor GitHub'a gider. FAIL varsa durulur.
2. **Kalibrasyon ve eğitim:** yalnız D-042 kabul edilip koddaki protokol "KABUL" olunca çalışır.
   - Kalibrasyon kol başına bir kez yapılır; epoch sayısını kural belirler.
   - Eğitim 3 tekrar × 5 fold'dur. Her fold bitince sonuç Drive'a yazılır. Colab kesilirse notebook'u yeniden çalıştırmak yeterli: bitmiş fold'lar atlanır. Checkpoint'ler Colab'ın yerel diskinde tutulur (Drive çöp kutusu kotayı doldurmasın diye): aynı oturumda yarım kalan fold kaldığı yerden sürer; makine değiştiyse o fold baştan başlar.

**Çalışma ortamı:** *Runtime → Change runtime type → T4 GPU*. Önce notebook 24 bitmiş ve raporu main'de olmalı.

## 0. Oturum kurulumu

In [11]:
import shutil, subprocess, platform, sys
print('Python:', sys.version.split()[0], '|', platform.platform())
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('UYARI: GPU yok (CPU oturumu). Bu notebook T4 GPU ister: Runtime → Change runtime type → T4 GPU.')

Python: 3.13.15 | Linux-6.6.122+-x86_64-with-glibc2.39
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07



In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib, codecs, platform
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
MODELS      = DRIVE_ROOT / 'models'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
LOCAL_CACHE = Path('/content/audio_cache_v1')
EMB_DIR     = DERIVED / 'embeddings_v1'
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
BACKBONES   = ['cnn10', 'cnn14', 'cnn14_16k', 'beats', 'wavlm_base_plus', 'wavlm_large']
EXPECTED_N_RECORDINGS, EXPECTED_N_PARTICIPANTS = 2393, 342   # Faz 1 önbellek kohortu (reports/audio_cache)
for p in [DERIVED / 'participant_context.csv', DERIVED / 'splits' / 'outer_r0.csv', DRIVE_CACHE / 'cache_info.json']:
    assert p.exists(), f'Bulunamadı: {p}'
for p in [MODELS, EXPERIMENTS, EMB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def sh(cmd, check=True):
    """Kabuk komutunu çalıştırır, çıktıyı canlı yazar. check=True iken komut başarısızsa hücreyi DURDURUR.
    (`!komut` başarısız olsa da hücre devam eder; 'Run all' hatayı fark etmeden sonraki adıma geçerdi.)"""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    while True:
        b = p.stdout.read1(1 << 14)
        if not b:
            break
        sys.stdout.write(dec.decode(b))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Komut başarısız (çıkış kodu {rc}): {cmd[:300]}')
    return None if check else rc


print('Girdiler mevcut.')

Girdiler mevcut.


In [14]:
# Repo: yoksa klonla; varsa GitHub'daki main'e güvenli biçimde geç (önceki çalıştırmanın rapor dalında kalmış olabilir)
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("GITHUB_TOKEN okunamadı: sol paneldeki anahtar (Secrets) simgesinden GITHUB_TOKEN ekli mi ve "
                       "'Notebook access' açık mı?") from e
assert token, 'GITHUB_TOKEN boş'
REMOTE = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'


def git(*args, check=True, show=False):
    """git komutu; çıktıda token gizlenir. check=True iken hata durumunda hücreyi durdurur."""
    r = subprocess.run(['git', '-C', str(REPO_DIR), *args], capture_output=True, text=True)
    r.stdout, r.stderr = r.stdout.replace(token, '***'), r.stderr.replace(token, '***')
    if show and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if check and r.returncode != 0:
        cmd = ' '.join(args[:3]).replace(token, '***')
        raise RuntimeError(f'git {cmd} başarısız (çıkış {r.returncode}):\n{(r.stdout + r.stderr).strip()}')
    return r


if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} var ama git deposu değil (yarım kalmış klon?). Şunu çalıştırıp bu hücreyi yeniden '
                       f'çalıştır:  !rm -rf {REPO_DIR}')
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', REMOTE, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone başarısız:\n' + (r.stdout + r.stderr).replace(token, '***'))
    print('Klonlandı.')
else:
    git('remote', 'set-url', 'origin', REMOTE)            # token yenilendiyse
    git('fetch', '--prune', 'origin')
    branch = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
    unpushed = git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip()
    dirty = git('status', '--porcelain', '--untracked-files=all').stdout.strip()
    if unpushed:
        raise RuntimeError(
            f"'{branch}' dalında GitHub'a gönderilmemiş commit var:\n{unpushed}\n"
            "Bu, yarım kalmış bir rapor gönderimidir; raporu kaybetmemek için önce 'Raporu GitHub'a gönder' hücresini "
            "çalıştır (aynı commit'i yeniden gönderir), sonra bu hücreyi yeniden çalıştır.")
    if dirty:
        at_main = branch == 'main' and git('rev-parse', 'HEAD').stdout == git('rev-parse', 'origin/main').stdout
        if not at_main:
            raise RuntimeError(
                f"Repo klasöründe commit edilmemiş dosyalar var (dal: {branch}):\n{dirty}\n"
                "Bunlar önceki bir çalıştırmanın raporlarıysa önce gönder hücresini çalıştır. Gerekmiyorsa klasörü silip "
                f"bu hücreyi yeniden çalıştır:  !rm -rf {REPO_DIR}")
        print('UYARI: main zaten GitHub ile aynı; önceki çalıştırmadan kalan commit edilmemiş dosyalara dokunulmadı:\n' + dirty)
    else:
        git('checkout', '-B', 'main', 'origin/main')       # temiz klasör: main = GitHub'daki main (yerel diğer dallar kalır)
        if branch != 'main':
            print(f"'{branch}' dalından main'e geçildi (o dal GitHub'da duruyor).")
assert git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip() == 'main'
COMMIT = git('rev-parse', 'HEAD').stdout.strip()
assert COMMIT == git('rev-parse', 'origin/main').stdout.strip(), 'yerel main GitHub main ile aynı değil'
print('Dal: main | commit:', COMMIT[:12], '(= origin/main)')

Dal: main | commit: 63795d66dfca (= origin/main)


In [15]:
sh(f'pip install -q -r {REPO_DIR}/requirements-colab.txt')
import torch, torchaudio, transformers, torchlibrosa, sklearn, numpy as np
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| transformers', transformers.__version__,
      '| sklearn', sklearn.__version__, '| numpy', np.__version__)
import torchaudio.compliance.kaldi  # BEATs bunu gerektirir; yoksa burada hata verir (sözleşme notu)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

torch 2.11.0+cu130 | torchaudio 2.11.0+cu130 | transformers 5.18.0 | sklearn 1.6.1 | numpy 2.1.3
CUDA: True Tesla T4


In [16]:
# Drive → yerel disk (okuma hızı). Yarım kalmış kopya .part olarak kalır ve bir sonraki çalıştırmada yeniden kopyalanır.
info = json.loads((DRIVE_CACHE / 'cache_info.json').read_text())
LOCAL_CACHE.mkdir(exist_ok=True)
for name in ['audio_32k.f32', 'audio_16k.f32', 'index.csv']:
    dst = LOCAL_CACHE / name
    if dst.exists() and sha256(dst) != info['files_sha256'][name]:
        print(name, ': yerel kopya bozuk, yeniden kopyalanıyor'); dst.unlink()
    if not dst.exists():
        tmp = dst.with_name(name + '.part')
        shutil.copy(DRIVE_CACHE / name, tmp)
        tmp.rename(dst)
    assert sha256(dst) == info['files_sha256'][name], f'{name}: sha256 cache_info.json ile uyuşmuyor (Drive kopyası bozuk?)'
print('Önbellek yerel diskte ve doğrulandı.')

Önbellek yerel diskte ve doğrulandı.


In [17]:
# Kapı: kod, tasarım ve notebook 24'ün çıktıları hazır
import pandas as pd
for p in ['docs/PHASE3_DESIGN.md', 'scripts/finetune.py', 'scripts/smoke_test_finetune.py', 'reports/phase3/G1_frozen_daygrouped.json',
          'reports/splits_day/splits_manifest.json']:
    if git('ls-files', '--error-unmatch', p, check=False).returncode != 0:
        raise RuntimeError(f'{p} main dalında yok: notebook 24 raporu ve Faz 3 PR\'ı birleştirilmiş mi?')
SPLITS_DAY = DERIVED / 'splits_day'
man = json.loads((REPO_DIR / 'reports/splits_day/splits_manifest.json').read_text())
for name, h in man['split_files_sha256'].items():
    assert sha256(SPLITS_DAY / name) == h, f'{name}: Drive split dosyası manifestle uyuşmuyor'
SEL = EXPERIMENTS / 'phase3' / 'frozen_probe_day' / 'd035_selection.csv'
sel = pd.read_csv(SEL)
assert len(sel) == 15
SSL_USED = sorted(set(sel.selected))
ARMS_ORDER = ['cnn10', 'cnn10_scratch'] + SSL_USED + ['cnn14']
FT_OUT = EXPERIMENTS / 'phase3' / 'finetune'
FT_CKPT = Path('/content/ft_ckpt')   # latest.pt yalnız yerel diskte (Drive'da silinen dosyalar çöpe gider ve kotadan sayılır)
FT_COMMON = (f'--splits-dir {SPLITS_DAY} --context {DERIVED}/participant_context.csv --cache-dir {LOCAL_CACHE} '
             f'--model-dir {MODELS} --out-root {FT_OUT} --ckpt-root {FT_CKPT}')
print('SSL kolunda D-035\'in seçtikleri:', sel.selected.value_counts().to_dict(), '| sıra:', ARMS_ORDER)

SSL kolunda D-035'in seçtikleri: {'beats': 12, 'wavlm_large': 2, 'wavlm_base_plus': 1} | sıra: ['cnn10', 'cnn10_scratch', 'beats', 'wavlm_base_plus', 'wavlm_large', 'cnn14']


## 1. SMK-002 — fine-tuning smoke testi (~15–30 dk)
Yalnız D-035'in en az bir fold'da seçtiği SSL modelleri sınanır. Rapor her durumda GitHub'a gönderilir (FAIL da korunur); FAIL varsa son hücre durur.

In [ ]:
rc = sh(f'cd {REPO_DIR} && python scripts/smoke_test_finetune.py --arms {",".join(ARMS_ORDER)} --splits-dir {SPLITS_DAY} '
        f'--context {DERIVED}/participant_context.csv --cache-dir {LOCAL_CACHE} --model-dir {MODELS} '
        f'--report-dir {REPO_DIR}/reports/phase3', check=False)
print('SMK-002 çıkış kodu:', rc)

In [ ]:
# Rapor commit'i her seferinde YENİ, zaman damgalı bir dala gider (aynı adlı dala ikinci gönderim reddedilirdi).
# Gönderim başarısız olursa commit yerel dalda kalır; bu hücreyi yeniden çalıştırmak aynı commit'i yeniden gönderir.
import re, time
import pandas as pd
BASE_BRANCH, PATHS, MSG = 'phase3-smk002', ['reports/phase3/SMK-002_finetune.md', 'reports/phase3/SMK-002_finetune.json'], 'Faz 3 SMK-002 fine-tuning smoke testi (Colab)'
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
cur = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
if cur.startswith(BASE_BRANCH + '-') and git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip():
    BRANCH = cur
    print('Önceki denemenin commit\'i yeniden gönderiliyor:', BRANCH)
else:
    assert cur == 'main', f"Beklenmeyen dal '{cur}': 0. bölümdeki repo hücresini yeniden çalıştır."
    git('add', '--', *PATHS)
    staged = git('diff', '--cached', '--name-only').stdout.split()
    assert staged, 'Gönderilecek değişiklik yok (rapor üretilmedi mi? yukarıdaki hücrenin çıktısına bak).'
    bad = [s for s in staged if not any(s == p or s.startswith(p.rstrip('/') + '/') for p in PATHS)]
    assert not bad, f'Beklenmeyen dosya: {bad}'
    ids = set(pd.read_csv(DERIVED / 'participant_context.csv').participant_id.astype(int))   # D-014 son kontrol
    for s in staged:
        txt = (REPO_DIR / s).read_text(errors='ignore')
        found = {int(m) for m in re.findall(r'(?<![\d.])(\d{6})(?![\d.])', txt)} & ids
        assert token not in txt and not found, f'{s}: token ya da katılımcı ID\'si içeriyor → commit edilmedi (D-014)'
    BRANCH = f"{BASE_BRANCH}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
    git('checkout', '-b', BRANCH)
    git('commit', '-m', MSG)
    print('commit:', git('rev-parse', '--short', 'HEAD').stdout.strip(), '| dosyalar:', staged)
r = git('push', '-u', 'origin', BRANCH, check=False, show=True)
if r.returncode != 0:
    raise RuntimeError(f"Push başarısız. Commit yerel '{BRANCH}' dalında duruyor (kaybolmadı). Token/ağ sorununu giderip "
                       "BU hücreyi yeniden çalıştır; aynı commit yeniden gönderilir.")
git('checkout', 'main')
print(f'Gönderildi: {BRANCH}\nPR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')

In [ ]:
assert rc == 0, 'SMK-002: en az bir kol FAIL → Bölüm 2\'ye geçme; raporu birlikte inceleyelim'
print('SMK-002 PASS. Bölüm 2 yalnız D-042 kabul edildikten sonra.')

## 1b. Kalibrasyon sonrası: CAL-001 ve DIAG-001 (D-043; ~10 dk, T4)
Kapıya tabi değil; sonuç üretmez.
- **CAL-001:** Drive'daki kalibrasyon eğrilerinden iki kuralın seçtiği epoch'lar (v1: kayıp, v2: AUC 1-SE). Eğitim yapılmaz.
- **DIAG-001:** CNN14 kalibrasyonundaki "sonlu olmayan logit" hatasının nedeni. 1 epoch eğitim, her adımda durum kontrolü, fp16 / fp32 değerlendirme, modül modül iz.
- İki rapor GitHub'a gönderilir. Okuma ve düzeltme D-043'te önceden yazılı eşlemeyle yapılır.

In [ ]:
sh(f'cd {REPO_DIR} && python scripts/finetune.py select --out-root {FT_OUT} --report-dir {REPO_DIR}/reports/phase3')

In [ ]:
rc_diag = sh(f'cd {REPO_DIR} && python scripts/diagnose_nonfinite.py --arm cnn14 --splits-dir {SPLITS_DAY} '
             f'--context {DERIVED}/participant_context.csv --cache-dir {LOCAL_CACHE} --model-dir {MODELS} '
             f'--report-dir {REPO_DIR}/reports/phase3', check=False)
print('DIAG-001 çıkış kodu:', rc_diag)

In [ ]:
# Rapor commit'i her seferinde YENİ, zaman damgalı bir dala gider (aynı adlı dala ikinci gönderim reddedilirdi).
# Gönderim başarısız olursa commit yerel dalda kalır; bu hücreyi yeniden çalıştırmak aynı commit'i yeniden gönderir.
import re, time
import pandas as pd
BASE_BRANCH, PATHS, MSG = 'phase3-diag', ['reports/phase3/DIAG-001_cnn14_nonfinite.md', 'reports/phase3/DIAG-001_cnn14_nonfinite.json', 'reports/phase3/CAL-001_epoch_selection.md'], 'Faz 3 DIAG-001 (CNN14 teşhisi) ve CAL-001 (kalibrasyon eğrileri) (Colab)'
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
cur = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
if cur.startswith(BASE_BRANCH + '-') and git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip():
    BRANCH = cur
    print('Önceki denemenin commit\'i yeniden gönderiliyor:', BRANCH)
else:
    assert cur == 'main', f"Beklenmeyen dal '{cur}': 0. bölümdeki repo hücresini yeniden çalıştır."
    git('add', '--', *PATHS)
    staged = git('diff', '--cached', '--name-only').stdout.split()
    assert staged, 'Gönderilecek değişiklik yok (rapor üretilmedi mi? yukarıdaki hücrenin çıktısına bak).'
    bad = [s for s in staged if not any(s == p or s.startswith(p.rstrip('/') + '/') for p in PATHS)]
    assert not bad, f'Beklenmeyen dosya: {bad}'
    ids = set(pd.read_csv(DERIVED / 'participant_context.csv').participant_id.astype(int))   # D-014 son kontrol
    for s in staged:
        txt = (REPO_DIR / s).read_text(errors='ignore')
        found = {int(m) for m in re.findall(r'(?<![\d.])(\d{6})(?![\d.])', txt)} & ids
        assert token not in txt and not found, f'{s}: token ya da katılımcı ID\'si içeriyor → commit edilmedi (D-014)'
    BRANCH = f"{BASE_BRANCH}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
    git('checkout', '-b', BRANCH)
    git('commit', '-m', MSG)
    print('commit:', git('rev-parse', '--short', 'HEAD').stdout.strip(), '| dosyalar:', staged)
r = git('push', '-u', 'origin', BRANCH, check=False, show=True)
if r.returncode != 0:
    raise RuntimeError(f"Push başarısız. Commit yerel '{BRANCH}' dalında duruyor (kaybolmadı). Token/ağ sorununu giderip "
                       "BU hücreyi yeniden çalıştır; aynı commit yeniden gönderilir.")
git('checkout', 'main')
print(f'Gönderildi: {BRANCH}\nPR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')

## 2. Kalibrasyon ve eğitim (yalnız protokol KABUL ise: D-042 + D-043)
Önce kapı: koddaki protokol "KABUL" olmalı, SMK-002 raporu main'de ve PASS olmalı. Sonra:
- **Kalibrasyon:** kol başına bir kez. Tekrar 0 / fold 0'ın iç ayrımında eğitilir; epoch sayısını kural seçer (D-043: iç doğrulama AUC'si, 1-SE; eğitim hücreleri epoch'u eğrilerden hesaplar). Sonuç Drive'da `calibration.json`; varsa atlanır.
- **Eğitim:** kollar sırayla. Her hücre bir kolun 15 fold'unu çalıştırır, bitenleri atlar.
- **Colab kesilirse:** 0. bölümü ve bu bölümün kapı hücresini çalıştırın, sonra kaldığınız kol hücresinden devam edin. Bitmiş fold'lar Drive'da, atlanır. Yarım kalan fold: aynı makinedeyse kaldığı epoch'tan sürer; makine değiştiyse (yerel disk silinir) aynı tohumla baştan başlar. En çok bir fold kaybedilir.
- **Kalibrasyon hücresi** bütün kolları sırayla kalibre eder (biteni atlar); kesilirse yeniden çalıştırın.

In [18]:
r = subprocess.run([sys.executable, '-c', 'import sys, json; sys.path.insert(0, "scripts"); import finetune as F; print(json.dumps(F.PHASE3))'],
                   cwd=REPO_DIR, capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-1500:]
PROT = json.loads(r.stdout.strip().splitlines()[-1])
assert PROT['status'] == 'KABUL', f"Faz 3 protokolü onaylı değil ({PROT['version']}: {PROT['status']}) → Bölüm 2 çalıştırılmaz"
smk = json.loads((REPO_DIR / 'reports/phase3/SMK-002_finetune.json').read_text())
ok = {x['arm']: x['status'] for x in smk['results']}
missing = [a for a in ARMS_ORDER if ok.get(a) != 'PASS']
assert not missing, f'SMK-002 PASS olmayan ya da sınanmamış kollar: {missing}'
print('Kapı geçti:', PROT['version'], '| kollar:', ARMS_ORDER)

Kapı geçti: Faz 3 protokolü v2 (D-042 + D-043) | kollar: ['cnn10', 'cnn10_scratch', 'beats', 'wavlm_base_plus', 'wavlm_large', 'cnn14']


In [19]:
for arm in ARMS_ORDER:
    sh(f'cd {REPO_DIR} && python scripts/finetune.py calibrate --arm {arm} {FT_COMMON}')
# D-043: epoch sayıları eğrilerden (v2 AUC 1-SE; v1 kayıp kuralı kayıt için yanında). CAL-001 raporu güncellenir.
sh(f'cd {REPO_DIR} && python scripts/finetune.py select --out-root {FT_OUT} --report-dir {REPO_DIR}/reports/phase3')

cnn10: kalibrasyon zaten var → 5 epoch
cnn10_scratch: kalibrasyon zaten var → 1 epoch
beats: kalibrasyon zaten var → 1 epoch
wavlm_base_plus: kalibrasyon zaten var → 1 epoch
wavlm_large: kalibrasyon zaten var → 1 epoch
cnn14 kalibrasyon: {'mode': 'calibrate', 'repeat': 0, 'fold': 0, 'n_train_participants': 216, 'n_eval_participants': 56, 'n_train_recordings': 1512, 'n_eval_recordings': 392, 'train_recordings_asthma_healthy': [1260, 252], 'eval_recordings_asthma_healthy': [322, 70], 'eval_participants_asthma_healthy': [46, 10], 'pos_weight': 0.2, 'n_days_train': 42, 'n_days_eval': 11}
  epoch=1 train_loss=0.2461 lr_backbone=0.0001 lr_head=0.001 steps=95 skipped_steps=1 epoch_s=16 elapsed_s=16 gpu_peak_gb=1.82 val_loss=0.2635 val_auc_monitor_only=0.413
  epoch=2 train_loss=0.2147 lr_backbone=0.0001 lr_head=0.001 steps=95 skipped_steps=0 epoch_s=13.9 elapsed_s=44.4 gpu_peak_gb=1.82 val_loss=0.3093 val_auc_monitor_only=0.6261
  epoch=3 train_loss=0.1981 lr_backbone=0.0001 lr_head=0.001 ste

In [20]:
sh(f'cd {REPO_DIR} && python scripts/finetune.py train --arm cnn10 {FT_COMMON}')

cnn10 r0 k0 (cnn10, 7 epoch): {'mode': 'cv', 'repeat': 0, 'fold': 0, 'n_train_participants': 272, 'n_eval_participants': 70, 'n_train_recordings': 1904, 'n_eval_recordings': 489, 'train_recordings_asthma_healthy': [1582, 322], 'eval_recordings_asthma_healthy': [398, 91], 'eval_participants_asthma_healthy': [57, 13], 'pos_weight': 0.2035, 'n_days_train': 53, 'n_days_eval': 13}
  epoch=1 train_loss=0.2311 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=12.5 elapsed_s=12.5 gpu_peak_gb=0.63
  epoch=2 train_loss=0.219 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=11 elapsed_s=23.6 gpu_peak_gb=0.63
  epoch=3 train_loss=0.208 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=10.8 elapsed_s=34.9 gpu_peak_gb=0.63
  epoch=4 train_loss=0.2031 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=11.2 elapsed_s=46.5 gpu_peak_gb=0.63
  epoch=5 train_loss=0.1944 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=11

In [21]:
sh(f'cd {REPO_DIR} && python scripts/finetune.py train --arm cnn10_scratch {FT_COMMON}')

cnn10_scratch r0 k0 (cnn10_scratch, 9 epoch): {'mode': 'cv', 'repeat': 0, 'fold': 0, 'n_train_participants': 272, 'n_eval_participants': 70, 'n_train_recordings': 1904, 'n_eval_recordings': 489, 'train_recordings_asthma_healthy': [1582, 322], 'eval_recordings_asthma_healthy': [398, 91], 'eval_participants_asthma_healthy': [57, 13], 'pos_weight': 0.2035, 'n_days_train': 53, 'n_days_eval': 13}
  epoch=1 train_loss=0.2363 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=11.9 elapsed_s=11.9 gpu_peak_gb=0.63
  epoch=2 train_loss=0.233 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=10.8 elapsed_s=22.9 gpu_peak_gb=0.63
  epoch=3 train_loss=0.2321 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=10.7 elapsed_s=34 gpu_peak_gb=0.63
  epoch=4 train_loss=0.2257 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=10.8 elapsed_s=45.2 gpu_peak_gb=0.63
  epoch=5 train_loss=0.2275 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_s

In [22]:
sh(f'cd {REPO_DIR} && python scripts/finetune.py train --arm ssl --selection {SEL} {FT_COMMON}')

ssl r0 k0 (beats, 8 epoch): {'mode': 'cv', 'repeat': 0, 'fold': 0, 'n_train_participants': 272, 'n_eval_participants': 70, 'n_train_recordings': 1904, 'n_eval_recordings': 489, 'train_recordings_asthma_healthy': [1582, 322], 'eval_recordings_asthma_healthy': [398, 91], 'eval_participants_asthma_healthy': [57, 13], 'pos_weight': 0.2035, 'n_days_train': 53, 'n_days_eval': 13}
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
  epoch=1 train_loss=0.2244 lr_backbone=1.008e-05 lr_head=0.0005042 steps=119 skipped_steps=0 epoch_s=47.7 elapsed_s=47.7 gpu_peak_gb=3.87
  epoch=2 train_loss=0.1854 lr_backbone=2e-05 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=42.9 elapsed_s=90.6 gpu_peak_gb=3.87
  epoch=3 train_loss=0.1559 lr_backbone=2e-05 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=43.3 elapsed_s=148.8 gpu_peak

In [23]:
sh(f'cd {REPO_DIR} && python scripts/finetune.py train --arm cnn14 {FT_COMMON}')

cnn14 r0 k0 (cnn14, 9 epoch): {'mode': 'cv', 'repeat': 0, 'fold': 0, 'n_train_participants': 272, 'n_eval_participants': 70, 'n_train_recordings': 1904, 'n_eval_recordings': 489, 'train_recordings_asthma_healthy': [1582, 322], 'eval_recordings_asthma_healthy': [398, 91], 'eval_participants_asthma_healthy': [57, 13], 'pos_weight': 0.2035, 'n_days_train': 53, 'n_days_eval': 13}
  epoch=1 train_loss=0.2411 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=33.1 elapsed_s=33.1 gpu_peak_gb=1.82
  epoch=2 train_loss=0.2204 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=18.3 elapsed_s=51.5 gpu_peak_gb=1.82
  epoch=3 train_loss=0.2034 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=1 epoch_s=18.8 elapsed_s=70.2 gpu_peak_gb=1.82
  epoch=4 train_loss=0.1824 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=0 epoch_s=18.7 elapsed_s=89 gpu_peak_gb=1.82
  epoch=5 train_loss=0.1572 lr_backbone=0.0001 lr_head=0.001 steps=119 skipped_steps=1 epoch_s=

In [24]:
# Durum: kol başına biten fold sayısı (15 beklenir)
rows = []
for arm in ['cnn10', 'cnn10_scratch', 'ssl', 'cnn14']:
    done = sorted(FT_OUT.glob(f'{arm}/r*_k*/fold.json'))
    rows.append({'kol': arm, 'biten fold': len(done),
                 'toplam dk': round(sum(json.loads(f.read_text())['total_s'] for f in done) / 60, 1)})
pd.DataFrame(rows)

,kol,biten fold,toplam dk
0,cnn10,15,23.2
1,cnn10_scratch,15,28.8
2,ssl,15,149.7
3,cnn14,15,55.2


## 3. Eğitim özetini GitHub'a gönder (kalibrasyon eğrileri ve fold özetleri; katılımcı ID'si yok)

In [25]:
dst = REPO_DIR / 'reports' / 'phase3' / 'finetune_runs'
dst.mkdir(parents=True, exist_ok=True)
for f in FT_OUT.glob('*/calibration/calibration.json'):
    shutil.copy(f, dst / f'{f.parent.parent.name}_calibration.json')
for f in FT_OUT.glob('*/r*_k*/fold.json'):
    shutil.copy(f, dst / f'{f.parent.parent.name}_{f.parent.name}.json')
print(len(list(dst.glob('*.json'))), 'dosya')

66 dosya


In [26]:
# Rapor commit'i her seferinde YENİ, zaman damgalı bir dala gider (aynı adlı dala ikinci gönderim reddedilirdi).
# Gönderim başarısız olursa commit yerel dalda kalır; bu hücreyi yeniden çalıştırmak aynı commit'i yeniden gönderir.
import re, time
import pandas as pd
BASE_BRANCH, PATHS, MSG = 'phase3-train', ['reports/phase3/finetune_runs', 'reports/phase3/CAL-001_epoch_selection.md'], 'Faz 3 eğitim özetleri: kalibrasyon ve fold kayıtları (Colab)'
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
cur = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
if cur.startswith(BASE_BRANCH + '-') and git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip():
    BRANCH = cur
    print('Önceki denemenin commit\'i yeniden gönderiliyor:', BRANCH)
else:
    assert cur == 'main', f"Beklenmeyen dal '{cur}': 0. bölümdeki repo hücresini yeniden çalıştır."
    git('add', '--', *PATHS)
    staged = git('diff', '--cached', '--name-only').stdout.split()
    assert staged, 'Gönderilecek değişiklik yok (rapor üretilmedi mi? yukarıdaki hücrenin çıktısına bak).'
    bad = [s for s in staged if not any(s == p or s.startswith(p.rstrip('/') + '/') for p in PATHS)]
    assert not bad, f'Beklenmeyen dosya: {bad}'
    ids = set(pd.read_csv(DERIVED / 'participant_context.csv').participant_id.astype(int))   # D-014 son kontrol
    for s in staged:
        txt = (REPO_DIR / s).read_text(errors='ignore')
        found = {int(m) for m in re.findall(r'(?<![\d.])(\d{6})(?![\d.])', txt)} & ids
        assert token not in txt and not found, f'{s}: token ya da katılımcı ID\'si içeriyor → commit edilmedi (D-014)'
    BRANCH = f"{BASE_BRANCH}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
    git('checkout', '-b', BRANCH)
    git('commit', '-m', MSG)
    print('commit:', git('rev-parse', '--short', 'HEAD').stdout.strip(), '| dosyalar:', staged)
r = git('push', '-u', 'origin', BRANCH, check=False, show=True)
if r.returncode != 0:
    raise RuntimeError(f"Push başarısız. Commit yerel '{BRANCH}' dalında duruyor (kaybolmadı). Token/ağ sorununu giderip "
                       "BU hücreyi yeniden çalıştır; aynı commit yeniden gönderilir.")
git('checkout', 'main')
print(f'Gönderildi: {BRANCH}\nPR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')

commit: 1be1087 | dosyalar: ['reports/phase3/CAL-001_epoch_selection.md', 'reports/phase3/finetune_runs/beats_calibration.json', 'reports/phase3/finetune_runs/cnn10_calibration.json', 'reports/phase3/finetune_runs/cnn10_r0_k0.json', 'reports/phase3/finetune_runs/cnn10_r0_k1.json', 'reports/phase3/finetune_runs/cnn10_r0_k2.json', 'reports/phase3/finetune_runs/cnn10_r0_k3.json', 'reports/phase3/finetune_runs/cnn10_r0_k4.json', 'reports/phase3/finetune_runs/cnn10_r1_k0.json', 'reports/phase3/finetune_runs/cnn10_r1_k1.json', 'reports/phase3/finetune_runs/cnn10_r1_k2.json', 'reports/phase3/finetune_runs/cnn10_r1_k3.json', 'reports/phase3/finetune_runs/cnn10_r1_k4.json', 'reports/phase3/finetune_runs/cnn10_r2_k0.json', 'reports/phase3/finetune_runs/cnn10_r2_k1.json', 'reports/phase3/finetune_runs/cnn10_r2_k2.json', 'reports/phase3/finetune_runs/cnn10_r2_k3.json', 'reports/phase3/finetune_runs/cnn10_r2_k4.json', 'reports/phase3/finetune_runs/cnn10_scratch_calibration.json', 'reports/phase3/fi